# 05. 위험도 계산 — SGIS 인구 조인

## 이 노트북이 하는 일
04의 415개 격자에 **SGIS 행정동 인구**(고령·독거)를 조인해, 위험도 = **발생확률(고령) × 목격불가(독거) × 도달지연**을 계산한다.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 SGIS OpenAPI인가:** 무료·즉시 발급, 행정동 단위 연령·가구 통계를 준다. 격자 단위 75+·1인가구는 집계구 파일(신청 대기)이 필요.
- **왜 65+를 역산하나:** SGIS OpenAPI는 연령 브래킷(75+)을 직접 안 준다. 대신 노령화지수(노인/유소년)와 노년부양비(노인/생산가능)를 주므로, 이 둘로 65+ 비율을 역산.
- **왜 평균 가구원수를 독거 프록시로 쓰나:** 1인가구 수를 직접 안 줘서. 평균 가구원수가 작을수록 독거가 많다는 근사(2.5 - 평균가구원).
- **왜 곱(×)인가:** 세 요소는 '모두 있어야 위험'이다. 하나라도 0에 가까우면(예: 젊은 동네) 위험도 0. AND 논리를 곱으로 표현.
- **왜 행정동→격자 균등배분인가:** 현재 데이터가 행정동 단위라, 같은 동 격자에 같은 밀도를 부여(1차 근사). 집계구 확보 시 정밀화.
- **왜 정규화(0~1)하나:** 세 요소의 단위가 달라(명/㎢, 가구원수, m) 그대로 곱할 수 없다. 각자 0~1로 맞춰 곱함.

## 데이터 출처
- 인구: SGIS OpenAPI(통계청). 경계·격자: 04. 도달지연: 04 그래프.

In [ ]:
import os, warnings                       # 폴더·경고
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, requests    # 수치·표·SGIS 호출
import geopandas as gpd                        # 지리 표
import osmnx as ox                             # 행정동 폴리곤 조회
import folium                                  # 위험지도
from shapely.ops import unary_union            # 폴리곤 합치기
from shapely.geometry import box               # 조회 사각형
from dotenv import load_dotenv                 # SGIS 키 로드
load_dotenv()                                  # .env 읽기
CRS_WGS, CRS_M = 4326, 5186                    # 위경도 / 평면
os.makedirs("outputs", exist_ok=True)

## 1. SGIS 인증 + 행정동 인구 조회

In [ ]:
ck=os.getenv("SGIS_CONSUMER_KEY"); cs=os.getenv("SGIS_CONSUMER_SECRET")               # .env의 SGIS 발급값 2개
BASE="https://sgisapi.kostat.go.kr/OpenAPI3"                                          # SGIS OpenAPI 공통 URL
tok=requests.get(f"{BASE}/auth/authentication.json",                                  # 키2개 → accessToken 발급(세션 토큰)
                 params={"consumer_key":ck,"consumer_secret":cs},timeout=15).json()["result"]["accessToken"]

DONGS={"21030510":"초량1동","21030520":"초량2동","21030530":"초량3동",                  # 대상 행정동 (SGIS 코드:이름)
       "21030550":"초량6동","21030700":"좌천동"}                                        # 부산 동구 SGIS 코드는 21030

def derive_65(tot, idx, per):                                                         # 65+ 역산: 노령화지수(O/C)·노년부양비(O/P)로
    tot,idx,per=float(tot),float(idx),float(per)                                      # O=노인, C=유소년, P=생산가능, tot=O+C+P
    return tot/(100/idx + 100/per + 1)                                               # O = tot / (C/O + P/O + 1) 를 정리한 식

pop={}                                                                               # 행정동별 지표 저장
for cd,nm in DONGS.items():                                                          # 각 행정동마다
    r=requests.get(f"{BASE}/stats/population.json",                                   # 인구 통계 조회
        params={"accessToken":tok,"year":"2023","adm_cd":cd,"low_search":"0"},timeout=20).json()["result"][0]
    pop[nm]={"tot":float(r["tot_ppltn"]),                                             # 총인구
             "p65":derive_65(r["tot_ppltn"],r["aged_child_idx"],r["oldage_suprt_per"]),  # 65+ 추정
             "avg_fmember":float(r["avg_fmember_cnt"])}                               # 평균 가구원수(독거 프록시 원자료)
pop_df=pd.DataFrame(pop).T                                                            # 표로
pop_df["solo_proxy"]=(2.5-pop_df["avg_fmember"]).clip(lower=0)                        # 독거 프록시 = 2.5-평균가구원(작을수록↑, 음수는 0)
print(pop_df.round(1))

## 2. 행정동 폴리곤(OSM) + A1 격자 로드

In [ ]:
DONGGU_BBOX=box(129.020,35.100,129.075,35.155)                                       # 동구 조회 사각형
adm=ox.features_from_polygon(DONGGU_BBOX, tags={"boundary":"administrative"})         # 행정경계 조회
adm=adm[adm.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()              # 면만
adm["name"]=adm["name"].astype(str)                                                  # 이름 문자열화
dong_poly=adm[adm["name"].isin(DONGS.values())][["name","geometry"]].dissolve("name").reset_index()  # 대상 5개 동 폴리곤만
dong_poly=gpd.GeoDataFrame(dong_poly, crs=CRS_WGS).to_crs(CRS_M)                      # 면적 계산 위해 5186
dong_poly["area_km2"]=dong_poly.area/1e6                                             # 각 동 면적(㎢) — 밀도 계산에 사용
print(dong_poly[["name","area_km2"]].round(3).to_string(index=False))

grid=gpd.read_parquet("outputs/grid_donggu_A1.parquet")                              # 04가 만든 격자(도달거리 포함) 로드
print("격자:", len(grid))

## 3. 격자 ↔ 행정동 조인 + 위험 3요소 계산

In [ ]:
cent=grid.copy(); cent["geometry"]=gpd.points_from_xy(grid["cx"],grid["cy"])          # 격자 중심점 도형 생성
cent=gpd.GeoDataFrame(cent, crs=CRS_M)
joined=gpd.sjoin(cent, dong_poly[["name","geometry"]], how="left", predicate="within")  # 중심점이 속한 행정동 공간조인
grid["dong"]=joined["name"].values                                                   # 각 격자에 소속 동 부여

dens={r["name"]: pop[r["name"]]["p65"]/r["area_km2"] for _,r in dong_poly.iterrows()} # 동별 65+ 밀도(명/㎢)
grid["pop65_dens"]=grid["dong"].map(dens)                                            # 발생확률 프록시 = 65+ 밀도(격자에 균등배분)
grid["solo_proxy"]=grid["dong"].map(lambda d: max(2.5-pop[d]["avg_fmember"],0.0) if d in pop else np.nan)  # 목격불가 프록시(독거)
grid["reach"]=grid["dist_station_m"]                                                 # 도달지연 = 안전센터 도로거리(04)

def nrm(s):                                                                          # 0~1 정규화(min-max)
    s=pd.to_numeric(s,errors="coerce")
    return (s-s.min())/(s.max()-s.min()) if s.max()>s.min() else s*0
grid["f_age"]=nrm(grid["pop65_dens"])                                                # 정규화한 발생확률
grid["f_solo"]=nrm(grid["solo_proxy"])                                               # 정규화한 목격불가
grid["f_reach"]=nrm(grid["reach"])                                                   # 정규화한 도달지연
grid["risk"]=grid["f_age"]*grid["f_solo"]*grid["f_reach"]                            # 위험도 = 세 요소 곱(AND)
grid["risk_norm"]=nrm(grid["risk"])                                                  # 위험도도 0~1 정규화(지도·MCLP용)
print("행정동별 격자 수:\n", grid["dong"].value_counts().to_string())
print("\n위험도 상위 행정동(격자 평균 risk):")
print(grid.groupby("dong")["risk"].mean().sort_values(ascending=False).round(3).to_string())

## 4. 위험지도 (folium)

In [ ]:
gw=grid.to_crs(CRS_WGS)                                                               # 지도용 위경도
def col(v):                                                                          # 위험도→색(빨강=고위험)
    if pd.isna(v): return "#dddddd"
    t=float(v)
    r=int(255*min(t*1.0+0.1,1)); g=int(200*(1-t)); b=int(60*(1-t))
    return f"#{r:02x}{g:02x}{b:02x}"
m=folium.Map(location=[35.122,129.045], zoom_start=14, tiles="cartodbpositron")
for _,r in gw.iterrows():                                                            # 격자별 위험도 색칠
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x,c=col(r["risk_norm"]):{"color":c,"weight":0.2,"fillColor":c,"fillOpacity":0.6},
        tooltip=f'{r["grid_id"]} {r["dong"]} risk={0 if pd.isna(r["risk_norm"]) else round(r["risk_norm"],2)}').add_to(m)
for _,r in dong_poly.to_crs(CRS_WGS).iterrows():                                     # 행정동 경계선
    folium.GeoJson(r["geometry"].__geo_interface__, style_function=lambda x:{"color":"#333","weight":1.5,"fill":False},
                   tooltip=r["name"]).add_to(m)
legend=('<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'  # 범례
        'border:1px solid #999;font-size:12px"><b>위험도</b> (고령×독거×도달지연)<br>'
        '<span style="color:#dd2222">&#9644;</span> 높음 &nbsp; <span style="color:#c8a000">&#9644;</span> 중간 '
        '&nbsp; <span style="color:#22aa22">&#9644;</span> 낮음</div>')
m.get_root().html.add_child(folium.Element(legend))
m.save("outputs/risk_map_A1.html")                                                    # 위험지도 저장
print("지도 저장: outputs/risk_map_A1.html")
m

## 5. 저장

In [ ]:
grid.to_parquet("outputs/grid_risk_A1.parquet")                                       # 위험도 격자 저장(06이 읽음)
try: grid.to_file("outputs/grid_risk_A1.gpkg", driver="GPKG")                         # QGIS용
except Exception as e: print("gpkg 경고:", e)
pop_df.to_csv("outputs/sgis_pop_donggu.csv", encoding="utf-8-sig")                     # SGIS 원자료 별도 저장(근거)
print("저장:", [f for f in sorted(os.listdir("outputs")) if "risk" in f or "sgis" in f])